In [0]:
%pip install openpyxl

In [0]:
%restart_python

In [0]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

SOURCE_FILE = Path("/Volumes/workspace/raw/rawvolume/rawdata/CustomerExtract.csv")
SPEC_FILE = Path("/Volumes/workspace/raw/rawvolume/rawdata/TransformationSpec.xlsx")
OUTPUT_FILE = Path("/Volumes/workspace/raw/rawvolume/rawdata/CustomerLoad.csv")

customer_df = pd.read_csv(SOURCE_FILE)
spec_df = pd.read_excel(SPEC_FILE, sheet_name="CustomerSpec")

In [0]:
print("Rows:", customer_df.shape[0])
print("Columns:", customer_df.shape[1])


In [0]:
customer_df.head()

In [0]:
customer_df.info()

In [0]:
customer_df.describe(include="all").T

**Null / completeness profiling** 

Clean DQ table

In [0]:
profile = pd.DataFrame({
    "Column": customer_df.columns,
    "Data_Type": customer_df.dtypes.astype(str).values,
    "Record_Count": len(customer_df),
    "Non_Null_Count": customer_df.notna().sum().values,
    "Null_Count": customer_df.isna().sum().values
})

profile["Null_%"] = (
    profile["Null_Count"] / profile["Record_Count"] * 100
).round(2)

profile

**Check duplicate Customer IDs**

KUNNR is the customer account number and should be unique for the source records.

In [0]:
duplicate_kunnr = customer_df[
    customer_df["KUNNR"].duplicated(keep=False)
]

print("Duplicate KUNNR records:", len(duplicate_kunnr))

In [0]:
customer_df["KUNNR"].isna().sum()

In [0]:
print(customer_df["KUNNR"])

**pecification-based DQ check**

Are there any fields in the extract that are populated with data that are not flagged in the spec as "Field Utilized in LEGACY System"?

So we compare:

CustomerExtract columns
             VS
Spec → Field Utilized in LEGACY System = Y

In [0]:
legacy_fields = set(
    spec_df.loc[
        spec_df["Field Utilized in LEGACY System"] == "Y",
        "SAP FIELD"
    ]
)

source_fields = set(customer_df.columns)

unexpected_fields = source_fields - legacy_fields

We only care about unexpected fields that actually contain data.

In [0]:
populated_unexpected_fields = [
    col for col in unexpected_fields
    if customer_df[col].notna().any()
]

**Required field validation**

We should identify required fields from the specification.

In [0]:
required_fields = spec_df[
    spec_df["NEW REQ"].astype(str).str.contains("REQ", na=False)
]["SAP FIELD"].dropna().tolist()

In [0]:
# For fields available in the source:

required_source_fields = [
    c for c in required_fields
    if c in customer_df.columns
]


required_nulls = customer_df[required_source_fields].isna().sum()

required_nulls[required_nulls > 0]

**Length validation**

The specification provides maximum lengths.

In [0]:
def check_length(df, field, max_length):
    if field not in df.columns:
        return 0
    
    return (
        df[field]
        .fillna("")
        .astype(str)
        .str.len()
        .gt(max_length)
        .sum()
    )

**Data cleansing**

Now we actually modify the data.

special characters in the Name 1 field

In [0]:
def clean_name(value):
    if pd.isna(value):
        return value

    value = str(value).strip()

    value = value.replace('"', '')

    value = re.sub(r"\s+", " ", value)

    return value

In [0]:
customer_df["NAME1"] = customer_df["NAME1"].apply(clean_name)

Standardize blank values

We should also normalize empty strings.

In [0]:
customer_df = customer_df.replace(r"^\s*$", np.nan, regex=True)

This is a normal ETL cleansing step.

In [0]:
for col in customer_df.select_dtypes(include="object").columns:
    customer_df[col] = customer_df[col].str.strip()

**Transformation layer** actual migration transformation.

**Create target dataframe**

Rather than modifying the original 183-column dataframe, I'd create a separate target dataframe.

In [0]:
target_df = pd.DataFrame()

In [0]:
target_df["KTOKD"] = customer_df["KTOKD"]
target_df["KUNNR"] = customer_df["KUNNR"]

target_df["BUKRS"] = "G100"
target_df["VKORG"] = "G100"
target_df["VTWEG"] = "20"
target_df["SPART"] = "10"

target_df["LAND1"] = customer_df["LAND1"]
target_df["NAME1"] = customer_df["NAME1"]
target_df["NAME2"] = customer_df["NAME2"]
target_df["ORT01"] = customer_df["ORT01"]
target_df["PSTLZ"] = customer_df["PSTLZ"]
target_df["REGIO"] = customer_df["REGIO"]

**Important SORTL transformation**

Sort field will be populated with the first 10 characters from the customer's name.

This is better than simply copying the source SORTL, because the specification explicitly defines the transformation.

In [0]:
target_df["SORTL"] = (
    target_df["NAME1"]
    .fillna("")
    .str[:10]
)

**Language transformation**

Both Systems use EN

Even though source contains SPRAS, the transformation rule takes precedence.



In [0]:
target_df["SPRAS"] = "EN"

ERDAT → DEF by System
ERNAM → DEF by System

we need to decide how to represent this in the flat file.

ERDAT and ERNAM are system-defined target fields and therefore are not derived from the legacy extract unless the target loading process requires explicit values.

If the required output schema expects those columns, we can keep them blank/NULL and clearly document why.

**Post-transformation DQ**

After creating target_df, run validation again.

Record count

In [0]:
assert len(target_df) == len(customer_df)

# Source records = 100
# Target records = 100

Duplicate customer IDs

In [0]:
target_df["KUNNR"].duplicated().sum()

In [0]:
print(target_df)

Required fields

In [0]:
target_df[required_target_fields].isna().sum()

Constant transformation checks

This gives very strong traceability.

In [0]:
assert target_df["BUKRS"].eq("G100").all()
assert target_df["VKORG"].eq("G100").all()
assert target_df["VTWEG"].eq("20").all()
assert target_df["SPART"].eq("10").all()
assert target_df["SPRAS"].eq("EN").all()

In [0]:
print(target_df)

**Reconciliation** -  Source and Target record count

In [0]:
source_record_count = len(customer_df)
target_record_count = len(target_df)

print("Source Record Count :", source_record_count)
print("Target Record Count :", target_record_count)

**calculate diference**

In [0]:
record_difference = source_record_count - target_record_count

print("Record Difference   :", record_difference)

if record_difference == 0:
    print("PASS - Source and target record counts match")
else:
    print("FAIL - Record count mismatch detected")

whether the target contains duplicate customer numbers.

In [0]:
source_duplicate_count = customer_df["KUNNR"].duplicated().sum()
target_duplicate_count = target_df["KUNNR"].duplicated().sum()

print("Source duplicate KUNNR :", source_duplicate_count)
print("Target duplicate KUNNR :", target_duplicate_count)

**Load CustomerLoad.csv**

In [0]:

target_df.to_csv(
    "/Volumes/workspace/raw/rawvolume/rawdata/CustomerLoad.csv",
    index=False
)

In [0]:
Final_output = pd.read_csv("/Volumes/workspace/raw/rawvolume/rawdata/CustomerLoad.csv")

print("Output file created successfully")
print("Rows:", len(Final_output))
print("Columns:", len(Final_output.columns))

**- Final output validation**

In [0]:
print("Source records       :", len(customer_df))
print("Target records       :", len(target_df))
print("Final output records :", len(Final_output))
print("Final output file    :", Final_output)

In [0]:
assert len(Final_output) == len(target_df)

print("PASS - Output file successfully validated")